<a href="https://colab.research.google.com/github/ldongheedev/-BDA-LLM-RAG-Program/blob/main/5%EC%A3%BC%EC%B0%A8_%EB%8F%84%EC%A0%84%EA%B3%BC%EC%A0%9C_%EC%BD%94%EB%9E%A9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
## 🏆 STEP 9. 도전 과제 — BERT로 영화 리뷰 분석

**목표:** BERT의 문맥 이해 능력을 직접 확인하고, 5주차 Word2Vec과 비교합니다.

> 💡 **현실 팁**
> BERT는 사전학습된 모델이므로 Word2Vec처럼 직접 학습할 필요가 없습니다.
> 대신 문맥을 이해하기 때문에 "연기가 훌륭하다"와 "연기가 별로다"를 구분할 수 있습니다.

### 🔧 준비: BERT 모델 로드

> ⬇️ 셀을 **순서대로** 실행하세요 (Shift+Enter)

In [1]:
# 필요한 도구 설치
!pip install transformers torch --quiet
print("✅ 설치 완료!")

✅ 설치 완료!


In [2]:
# 필요한 도구 불러오기 + BERT 모델 로드
from transformers import AutoTokenizer, AutoModel, pipeline
import torch
import numpy as np

model_name = "klue/bert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModel.from_pretrained(model_name)

# 문장을 BERT 벡터로 변환하는 함수
def 문장_벡터(text):
    inputs = tokenizer(text, return_tensors="pt",
                       truncation=True, max_length=128, padding=True)
    with torch.no_grad():
        outputs = bert_model(**inputs)
    return outputs.last_hidden_state[:, 0, :].squeeze().numpy()

# 코사인 유사도 함수
def 코사인(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# 빈칸 채우기 파이프라인
fill_mask = pipeline("fill-mask", model="klue/bert-base")

# 감성 분류 파이프라인
classifier = pipeline("sentiment-analysis", model="snunlp/KR-FinBert-SC")

print(f"✅ 준비 완료! 모델: {model_name}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/425 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/372 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

✅ 준비 완료! 모델: klue/bert-base


---
### 과제 1. 문맥에 따른 벡터 차이 확인

Word2Vec에서는 "연기"가 항상 같은 벡터였습니다.
BERT는 문맥에 따라 **다른 벡터**를 만드는지 직접 확인합니다.

In [3]:
# 같은 단어 "연기"가 문맥에 따라 다른 벡터를 갖는지 확인
과제1_문장들 = [
    "이 영화의 연기가 훌륭하다",         # 연기 = acting (긍정)
    "이 영화의 연기가 별로다",           # 연기 = acting (부정)
    "산에서 연기가 피어오르고 있다",      # 연기 = smoke
    # ✏️ 여기에 직접 문장을 추가해보세요!
    # "여기에 문장 입력",
]

벡터들 = [문장_벡터(s) for s in 과제1_문장들]

print("문맥에 따른 유사도 비교:")
print("-" * 55)
for i in range(len(과제1_문장들)):
    for j in range(i+1, len(과제1_문장들)):
        sim = 코사인(벡터들[i], 벡터들[j])
        막대 = "█" * int(sim * 20)
        print(f"  {sim:.4f}  {막대}")
        print(f"    '{과제1_문장들[i]}'")
        print(f"    '{과제1_문장들[j]}'")
        print()

print("→ Word2Vec에서는 '연기 훌륭' ≈ '연기 별로' (유사도 0.95+)")
print("→ BERT에서는 문맥이 다르면 유사도가 낮아집니다 ✅")

문맥에 따른 유사도 비교:
-------------------------------------------------------
  0.8595  █████████████████
    '이 영화의 연기가 훌륭하다'
    '이 영화의 연기가 별로다'

  0.7284  ██████████████
    '이 영화의 연기가 훌륭하다'
    '산에서 연기가 피어오르고 있다'

  0.6974  █████████████
    '이 영화의 연기가 별로다'
    '산에서 연기가 피어오르고 있다'

→ Word2Vec에서는 '연기 훌륭' ≈ '연기 별로' (유사도 0.95+)
→ BERT에서는 문맥이 다르면 유사도가 낮아집니다 ✅


---
### 과제 2. 빈칸 채우기로 BERT의 문맥 이해 확인

BERT는 **빈칸 채우기(MLM)** 로 학습되었습니다.
문맥에 맞는 단어를 잘 예측하는지 확인합니다.

In [4]:
# BERT 빈칸 채우기 테스트
과제2_문장들 = [
    "배우의 [MASK]가 정말 훌륭했어요",
    "스토리가 [MASK]하고 각본이 허술했어요",
    "왕의 [MASK]가 압도적이었다",
    "영상미가 [MASK] 촬영이 훌륭하다",
    # ✏️ 여기에 직접 빈칸 문장을 추가해보세요!
    # "[MASK]가 있는 문장",
]

print("BERT 빈칸 채우기 결과:")
print("-" * 55)
for 문장 in 과제2_문장들:
    결과 = fill_mask(문장)
    print(f"  문장: '{문장}'")
    for r in 결과[:3]:
        막대 = "█" * int(r['score'] * 20)
        print(f"    → '{r['token_str']}': {r['score']:.4f}  {막대}")
    print()

BERT 빈칸 채우기 결과:
-------------------------------------------------------
  문장: '배우의 [MASK]가 정말 훌륭했어요'
    → '노래': 0.0426  
    → '연기': 0.0321  
    → '키스': 0.0301  

  문장: '스토리가 [MASK]하고 각본이 허술했어요'
    → '너무': 0.2784  █████
    → '뻔': 0.1388  ██
    → '[UNK]': 0.0675  █

  문장: '왕의 [MASK]가 압도적이었다'
    → '분노': 0.0688  █
    → '목소리': 0.0613  █
    → '[UNK]': 0.0607  █

  문장: '영상미가 [MASK] 촬영이 훌륭하다'
    → '좋아': 0.5255  ██████████
    → '뛰어나': 0.0595  █
    → '[UNK]': 0.0414  



---
### 과제 3. BERT 감성 분류

5주차에서는 Word2Vec → 문서_벡터 → LogisticRegression으로 분류했습니다.
BERT는 사전학습된 분류 모델을 **바로** 사용할 수 있습니다.

In [5]:
# 감성 분류 테스트
과제3_리뷰들 = [
    "배우의 연기가 정말 훌륭했어요",
    "스토리가 지루하고 각본이 허술했어요",
    "영상미가 아름답고 OST가 감동적이에요",
    "기대했는데 실망이에요 시간 아깝다",
    "주인공의 카리스마가 압도적이었다",
    "최악이에요 졸렸다 별로",
    "왕 역할 배우의 연기에 눈물이 났다",
    # ✏️ 여기에 직접 리뷰를 추가해보세요!
    # "여기에 리뷰 입력",
]

print("BERT 감성 분류 결과:")
print("-" * 60)
for 리뷰 in 과제3_리뷰들:
    결과 = classifier(리뷰)[0]
    라벨 = "긍정 ✅" if 결과['label'] == 'positive' else "부정 ❌"
    막대 = "█" * int(결과['score'] * 20)
    print(f"  {라벨} ({결과['score']:.4f})  {막대}")
    print(f"    '{리뷰}'")
    print()

BERT 감성 분류 결과:
------------------------------------------------------------
  부정 ❌ (0.9945)  ███████████████████
    '배우의 연기가 정말 훌륭했어요'

  부정 ❌ (0.9977)  ███████████████████
    '스토리가 지루하고 각본이 허술했어요'

  부정 ❌ (0.9993)  ███████████████████
    '영상미가 아름답고 OST가 감동적이에요'

  부정 ❌ (0.7110)  ██████████████
    '기대했는데 실망이에요 시간 아깝다'

  부정 ❌ (0.9988)  ███████████████████
    '주인공의 카리스마가 압도적이었다'

  부정 ❌ (0.9661)  ███████████████████
    '최악이에요 졸렸다 별로'

  부정 ❌ (0.9344)  ██████████████████
    '왕 역할 배우의 연기에 눈물이 났다'



---
### 과제 4. 유사 리뷰 검색

5주차의 `model.wv.most_similar("연기")` 는 유사 **단어**를 검색했습니다.
BERT는 유사 **문장**을 검색할 수 있습니다.

In [6]:
# 유사 리뷰 검색 시스템
전체_리뷰 = [
    "주인공의 연기가 압도적이었다",
    "영상미가 정말 아름다웠어요",
    "스토리 전개가 너무 느렸다",
    "배우들의 호흡이 완벽했다",
    "OST가 귀에 맴돈다 감동적이다",
    "기대했는데 좀 실망이에요",
    "역사적 고증이 잘 되어있다",
    "재관람할 의향이 있어요 추천",
    "왕 역할 배우의 카리스마가 대단하다",
    "눈물이 나올 정도로 감동적이었다",
    "지루해서 중간에 잠이 들었다",
    "영상 색감이 환상적이다",
]

# 리뷰 벡터 미리 계산
리뷰_벡터들 = [문장_벡터(r) for r in 전체_리뷰]

def 유사_리뷰_검색(query, top_k=3):
    query_vec = 문장_벡터(query)
    유사도_목록 = [(i, 코사인(query_vec, v)) for i, v in enumerate(리뷰_벡터들)]
    유사도_목록.sort(key=lambda x: x[1], reverse=True)

    print(f"검색어: '{query}'")
    print("-" * 50)
    for rank, (idx, sim) in enumerate(유사도_목록[:top_k], 1):
        막대 = "█" * int(sim * 20)
        print(f"  {rank}위: {sim:.4f}  {막대}")
        print(f"       '{전체_리뷰[idx]}'")
    print()

# 테스트 검색
검색어들 = [
    "연기가 좋았어요",
    "영화가 지루했다",
    "화면이 예뻤다",
    # ✏️ 여기에 직접 검색어를 추가해보세요!
    # "검색어 입력",
]

for q in 검색어들:
    유사_리뷰_검색(q)

검색어: '연기가 좋았어요'
--------------------------------------------------
  1위: 0.8698  █████████████████
       '영상미가 정말 아름다웠어요'
  2위: 0.8510  █████████████████
       '배우들의 호흡이 완벽했다'
  3위: 0.8456  ████████████████
       '눈물이 나올 정도로 감동적이었다'

검색어: '영화가 지루했다'
--------------------------------------------------
  1위: 0.8471  ████████████████
       '스토리 전개가 너무 느렸다'
  2위: 0.8357  ████████████████
       '지루해서 중간에 잠이 들었다'
  3위: 0.7991  ███████████████
       'OST가 귀에 맴돈다 감동적이다'

검색어: '화면이 예뻤다'
--------------------------------------------------
  1위: 0.9197  ██████████████████
       '영상 색감이 환상적이다'
  2위: 0.8691  █████████████████
       '영상미가 정말 아름다웠어요'
  3위: 0.8550  █████████████████
       'OST가 귀에 맴돈다 감동적이다'



---
### 과제 5. TF-IDF vs Word2Vec vs BERT 비교

동일한 리뷰 데이터에서 세 가지 방법의 유사도를 비교합니다.

In [7]:
# 세 가지 방법 비교
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

리뷰1 = "배우의 연기가 정말 훌륭했어요"
리뷰2 = "주인공의 연기력이 정말 뛰어났어요"    # 같은 뜻
리뷰3 = "스토리가 지루하고 각본이 허술했어요"  # 다른 뜻

# TF-IDF 유사도
tfidf_vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(2,4))
X_tfidf = tfidf_vec.fit_transform([리뷰1, 리뷰2, 리뷰3])
tfidf_sim = cosine_similarity(X_tfidf)

# BERT 유사도
v1 = 문장_벡터(리뷰1)
v2 = 문장_벡터(리뷰2)
v3 = 문장_벡터(리뷰3)

print("=" * 60)
print("TF-IDF vs Word2Vec vs BERT 유사도 비교")
print("=" * 60)
print(f"{'비교':32}  {'TF-IDF':8}  {'BERT'}")
print("-" * 60)
print(f"배우연기 ↔ 주인공연기력 (같은 뜻)  {tfidf_sim[0][1]:8.4f}  {코사인(v1,v2):.4f}")
print(f"배우연기 ↔ 스토리지루함 (다른 뜻)  {tfidf_sim[0][2]:8.4f}  {코사인(v1,v3):.4f}")
print()
print("TF-IDF: 같은 뜻도, 다른 뜻도 비슷하게 낮음 ❌")
print("BERT:   같은 뜻은 높고, 다른 뜻은 낮음 ✅")
print()
print("💡 5주차 Word2Vec 결과와도 비교해보세요!")
print("   Word2Vec: 배우연기↔주인공연기력 0.98 / 배우연기↔스토리지루함 0.45")

TF-IDF vs Word2Vec vs BERT 유사도 비교
비교                                TF-IDF    BERT
------------------------------------------------------------
배우연기 ↔ 주인공연기력 (같은 뜻)    0.1975  0.9471
배우연기 ↔ 스토리지루함 (다른 뜻)    0.0908  0.7999

TF-IDF: 같은 뜻도, 다른 뜻도 비슷하게 낮음 ❌
BERT:   같은 뜻은 높고, 다른 뜻은 낮음 ✅

💡 5주차 Word2Vec 결과와도 비교해보세요!
   Word2Vec: 배우연기↔주인공연기력 0.98 / 배우연기↔스토리지루함 0.45


---
### 📌 과제 결과 확인 포인트

| 과제 | 확인할 것 |
|------|------|
| 과제 1 | "연기 훌륭" ↔ "연기 별로" 유사도가 1.0보다 확실히 낮은지 (Word2Vec에서는 0.95+였음) |
| 과제 2 | BERT가 문맥에 맞는 단어를 잘 예측하는지 ("배우의 [MASK]" → "연기") |
| 과제 3 | 긍정 리뷰는 positive, 부정 리뷰는 negative로 분류되는지 |
| 과제 4 | 검색어와 의미적으로 비슷한 리뷰가 상위에 나오는지 |
| 과제 5 | TF-IDF < Word2Vec < BERT 순으로 의미 구분이 잘 되는지 |

> 💡 **5주차 → 9주차 핵심 변화**
>
> ```
> 5주차: model.wv.most_similar("연기")     → 유사 '단어' 검색
> 9주차: 유사_리뷰_검색("연기가 좋았어요") → 유사 '문장' 검색
>
> 5주차: "연기 훌륭" ≈ "연기 별로" (문맥 무시)
> 9주차: "연기 훌륭" ≠ "연기 별로" (문맥 이해 ✅)
> ```

In [8]:
# ✏️ 자유 실험 공간 — 직접 테스트해보세요!
